# Phase 1: Exploratory Data Analysis (EDA)

**Project:** Time Series Forecasting with Clustering  
**Course:** Recent Developments in KDD (SS 2026)  
**Date:** March 12, 2026

---

## Objectives

1. **Data Loading & Validation** - Load and verify data quality
2. **Descriptive Statistics** - Compute household-level statistics
3. **Temporal Pattern Analysis** - Identify seasonality, trends, cycles
4. **Initial Clustering Insights** - Preliminary grouping patterns

---

## 0. Setup & Imports

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# Project imports
import sys
sys.path.append('..')
from src.utils.data_loader import (
    load_train_data, 
    load_test_data, 
    load_both_datasets, 
    validate_data
)
from src.preprocessing.eda_analysis import (
    compute_household_statistics,
    detect_outlier_households,
    aggregate_by_day_of_week,
    aggregate_by_month,
    detect_seasonality,
    compute_autocorrelation,
    categorize_households,
    detect_trend
)
from src.utils.config import RANDOM_SEED

# Set random seed for reproducibility
np.random.seed(RANDOM_SEED)

print("✅ All imports successful!")

---
## 1. Data Loading & Validation

### Task 1.1.1 - 1.1.5: Load and verify data quality

In [ ]:
# Load both datasets
print("Loading datasets...")
train_data, test_data = load_both_datasets()

print("\n" + "="*80)
print("DATASET OVERVIEW")
print("="*80)
print(f"\nTraining Data (2023):")
print(f"  Shape: {train_data.shape}")
print(f"  Households: {len(train_data)}")
print(f"  Days: {train_data.shape[1]}")
print(f"  Total observations: {train_data.size:,}")

print(f"\nTest Data (2024):")
print(f"  Shape: {test_data.shape}")
print(f"  Households: {len(test_data)}")
print(f"  Days: {test_data.shape[1]}")
print(f"  Total observations: {test_data.size:,}")

In [ ]:
# Validate data quality
print("\n" + "="*80)
print("DATA QUALITY CHECKS")
print("="*80)

train_validation = validate_data(train_data, "Training")
test_validation = validate_data(test_data, "Test")

# Display validation results
print("\n📊 Training Data (2023) Validation:")
for key, value in train_validation.items():
    print(f"  {key}: {value}")

print("\n📊 Test Data (2024) Validation:")
for key, value in test_validation.items():
    print(f"  {key}: {value}")

In [ ]:
# Check data types and value ranges
print("\n" + "="*80)
print("VALUE RANGES")
print("="*80)

print("\nTraining Data (2023):")
print(f"  Min: {train_data.min().min():.4f} kWh")
print(f"  Max: {train_data.max().max():.4f} kWh")
print(f"  Mean: {train_data.mean().mean():.4f} kWh")
print(f"  Median: {train_data.median().median():.4f} kWh")
print(f"  Std: {train_data.std().std():.4f} kWh")

print("\nTest Data (2024):")
print(f"  Min: {test_data.min().min():.4f} kWh")
print(f"  Max: {test_data.max().max():.4f} kWh")
print(f"  Mean: {test_data.mean().mean():.4f} kWh")
print(f"  Median: {test_data.median().median():.4f} kWh")
print(f"  Std: {test_data.std().std():.4f} kWh")

In [ ]:
# Verify household IDs match
print("\n" + "="*80)
print("HOUSEHOLD ID VERIFICATION")
print("="*80)

train_ids = set(train_data.index)
test_ids = set(test_data.index)

print(f"\nTrain IDs: {len(train_ids)}")
print(f"Test IDs: {len(test_ids)}")
print(f"IDs match: {train_ids == test_ids}")

if train_ids != test_ids:
    print(f"\n⚠️ WARNING: {len(train_ids - test_ids)} IDs in train but not in test")
    print(f"⚠️ WARNING: {len(test_ids - train_ids)} IDs in test but not in train")
else:
    print("\n✅ All household IDs match perfectly!")

In [ ]:
# Document data quality issues
print("\n" + "="*80)
print("DATA QUALITY SUMMARY")
print("="*80)

quality_issues = []

if train_validation['missing_values'] > 0:
    quality_issues.append(f"Missing values in training data: {train_validation['missing_values']}")

if test_validation['missing_values'] > 0:
    quality_issues.append(f"Missing values in test data: {test_validation['missing_values']}")

if train_validation['zero_values'] > 0:
    zero_pct_train = (train_validation['zero_values'] / train_data.size) * 100
    quality_issues.append(f"Zero consumption values in training: {train_validation['zero_values']:,} ({zero_pct_train:.2f}%)")

if test_validation['zero_values'] > 0:
    zero_pct_test = (test_validation['zero_values'] / test_data.size) * 100
    quality_issues.append(f"Zero consumption values in test: {test_validation['zero_values']:,} ({zero_pct_test:.2f}%)")

if quality_issues:
    print("\n⚠️ Issues Found:")
    for issue in quality_issues:
        print(f"  • {issue}")
    print("\n📝 Note: Zero values may represent vacant properties or equipment failures.")
    print("   These should be investigated and handled in preprocessing phase.")
else:
    print("\n✅ No data quality issues detected!")

---
## 2. Descriptive Statistics

### Task 1.2.1 - 1.2.5: Per-household statistics and outlier detection

In [ ]:
# Compute per-household statistics
print("Computing household statistics...")
household_stats = compute_household_statistics(train_data)

print("\n" + "="*80)
print("HOUSEHOLD STATISTICS (Training Data 2023)")
print("="*80)
print(household_stats.describe())

In [ ]:
# Visualize overall consumption distribution
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Household Consumption Statistics Distribution', fontsize=16, fontweight='bold')

# Mean consumption
axes[0, 0].hist(household_stats['mean'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(household_stats['mean'].median(), color='red', linestyle='--', label=f'Median: {household_stats["mean"].median():.2f}')
axes[0, 0].set_xlabel('Mean Consumption (kWh/day)')
axes[0, 0].set_ylabel('Number of Households')
axes[0, 0].set_title('Mean Daily Consumption')
axes[0, 0].legend()

# Standard deviation
axes[0, 1].hist(household_stats['std'], bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].axvline(household_stats['std'].median(), color='red', linestyle='--', label=f'Median: {household_stats["std"].median():.2f}')
axes[0, 1].set_xlabel('Standard Deviation (kWh)')
axes[0, 1].set_ylabel('Number of Households')
axes[0, 1].set_title('Consumption Variability')
axes[0, 1].legend()

# Coefficient of variation
axes[0, 2].hist(household_stats['cv'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0, 2].axvline(household_stats['cv'].median(), color='red', linestyle='--', label=f'Median: {household_stats["cv"].median():.2f}')
axes[0, 2].set_xlabel('Coefficient of Variation (CV)')
axes[0, 2].set_ylabel('Number of Households')
axes[0, 2].set_title('Relative Variability (CV = std/mean)')
axes[0, 2].legend()

# Min-Max range
axes[1, 0].hist(household_stats['range'], bins=50, edgecolor='black', alpha=0.7, color='purple')
axes[1, 0].axvline(household_stats['range'].median(), color='red', linestyle='--', label=f'Median: {household_stats["range"].median():.2f}')
axes[1, 0].set_xlabel('Consumption Range (kWh)')
axes[1, 0].set_ylabel('Number of Households')
axes[1, 0].set_title('Min-Max Range')
axes[1, 0].legend()

# Zero consumption percentage
axes[1, 1].hist(household_stats['zero_pct'], bins=50, edgecolor='black', alpha=0.7, color='red')
axes[1, 1].axvline(household_stats['zero_pct'].median(), color='darkred', linestyle='--', label=f'Median: {household_stats["zero_pct"].median():.2f}%')
axes[1, 1].set_xlabel('Zero Consumption Days (%)')
axes[1, 1].set_ylabel('Number of Households')
axes[1, 1].set_title('Zero-Consumption Periods')
axes[1, 1].legend()

# Skewness
axes[1, 2].hist(household_stats['skew'], bins=50, edgecolor='black', alpha=0.7, color='teal')
axes[1, 2].axvline(household_stats['skew'].median(), color='red', linestyle='--', label=f'Median: {household_stats["skew"].median():.2f}')
axes[1, 2].axvline(0, color='black', linestyle='-', linewidth=1, alpha=0.5, label='Symmetric')
axes[1, 2].set_xlabel('Skewness')
axes[1, 2].set_ylabel('Number of Households')
axes[1, 2].set_title('Distribution Skewness')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Identify outlier households
print("\n" + "="*80)
print("OUTLIER DETECTION")
print("="*80)

high_outliers_iqr, low_outliers_iqr = detect_outlier_households(household_stats, method='iqr', threshold=1.5)
high_outliers_z, low_outliers_z = detect_outlier_households(household_stats, method='zscore', threshold=3.0)

print(f"\nIQR Method (threshold=1.5):")
print(f"  High consumption outliers: {len(high_outliers_iqr)} households")
print(f"  Low consumption outliers: {len(low_outliers_iqr)} households")

print(f"\nZ-Score Method (threshold=3.0):")
print(f"  High consumption outliers: {len(high_outliers_z)} households")
print(f"  Low consumption outliers: {len(low_outliers_z)} households")

# Show examples of extreme households
print(f"\n📊 Top 5 Highest Consumers:")
top5 = household_stats.nlargest(5, 'mean')[['mean', 'std', 'min', 'max', 'cv']]
print(top5)

print(f"\n📊 Top 5 Lowest Consumers:")
bottom5 = household_stats.nsmallest(5, 'mean')[['mean', 'std', 'min', 'max', 'cv']]
print(bottom5)

In [ ]:
# Box plots for consumption patterns
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Household Consumption Patterns (Outlier Visualization)', fontsize=16, fontweight='bold')

# Box plot of mean consumption
axes[0].boxplot(household_stats['mean'], vert=True)
axes[0].set_ylabel('Mean Daily Consumption (kWh)')
axes[0].set_title('Mean Consumption Distribution with Outliers')
axes[0].grid(True, alpha=0.3)

# Scatter plot: Mean vs CV
axes[1].scatter(household_stats['mean'], household_stats['cv'], alpha=0.5, s=10)
axes[1].set_xlabel('Mean Consumption (kWh/day)')
axes[1].set_ylabel('Coefficient of Variation')
axes[1].set_title('Variability vs Mean Consumption')
axes[1].grid(True, alpha=0.3)

# Highlight outliers
if high_outliers_iqr:
    high_data = household_stats.loc[high_outliers_iqr]
    axes[1].scatter(high_data['mean'], high_data['cv'], color='red', s=30, alpha=0.7, label='High Outliers')

if low_outliers_iqr:
    low_data = household_stats.loc[low_outliers_iqr]
    axes[1].scatter(low_data['mean'], low_data['cv'], color='blue', s=30, alpha=0.7, label='Low Outliers')

axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Detect zero-consumption periods
print("\n" + "="*80)
print("ZERO-CONSUMPTION ANALYSIS")
print("="*80)

households_with_zeros = household_stats[household_stats['zero_days'] > 0]
print(f"\nHouseholds with at least one zero day: {len(households_with_zeros)} ({len(households_with_zeros)/len(household_stats)*100:.2f}%)")

# Categorize by zero percentage
zero_categories = {
    'Rare (< 5%)': len(household_stats[(household_stats['zero_pct'] > 0) & (household_stats['zero_pct'] < 5)]),
    'Occasional (5-20%)': len(household_stats[(household_stats['zero_pct'] >= 5) & (household_stats['zero_pct'] < 20)]),
    'Frequent (20-50%)': len(household_stats[(household_stats['zero_pct'] >= 20) & (household_stats['zero_pct'] < 50)]),
    'Majority (> 50%)': len(household_stats[household_stats['zero_pct'] >= 50])
}

print("\nZero-Consumption Frequency:")
for category, count in zero_categories.items():
    print(f"  {category}: {count} households")

---
## 3. Temporal Pattern Analysis

### Task 1.3.1 - 1.3.6: Time series patterns, seasonality, trends

In [ ]:
# Plot sample time series (10-20 random households)
np.random.seed(RANDOM_SEED)
sample_households = np.random.choice(train_data.index, size=15, replace=False)

fig, axes = plt.subplots(5, 3, figsize=(20, 15))
fig.suptitle('Sample Time Series: 15 Random Households (2023)', fontsize=16, fontweight='bold')

for idx, hh_id in enumerate(sample_households):
    row = idx // 3
    col = idx % 3
    
    series = train_data.loc[hh_id]
    axes[row, col].plot(series.values, linewidth=0.8)
    axes[row, col].set_title(f'Household {hh_id} (mean={household_stats.loc[hh_id, "mean"]:.2f} kWh)')
    axes[row, col].set_xlabel('Day of Year')
    axes[row, col].set_ylabel('Consumption (kWh)')
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Aggregate consumption by day of week
print("\n" + "="*80)
print("DAY-OF-WEEK ANALYSIS")
print("="*80)

dow_data = aggregate_by_day_of_week(train_data)

# Average consumption by day of week (across all households)
dow_mean = dow_data.mean(axis=0)
print("\nAverage Consumption by Day of Week:")
print(dow_mean)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot
axes[0].bar(dow_mean.index, dow_mean.values, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Day of Week')
axes[0].set_ylabel('Average Consumption (kWh/day)')
axes[0].set_title('Average Daily Consumption by Day of Week')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].tick_params(axis='x', rotation=45)

# Box plot (distribution across households)
dow_data.boxplot(ax=axes[1])
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Consumption (kWh/day)')
axes[1].set_title('Consumption Distribution by Day of Week')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Weekday vs Weekend comparison
weekday_cols = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
weekend_cols = ['Saturday', 'Sunday']

household_stats['weekday_mean'] = dow_data[weekday_cols].mean(axis=1)
household_stats['weekend_mean'] = dow_data[weekend_cols].mean(axis=1)
household_stats['weekend_ratio'] = household_stats['weekend_mean'] / (household_stats['weekday_mean'] + 1e-10)

print("\n" + "="*80)
print("WEEKDAY vs WEEKEND PATTERNS")
print("="*80)
print(f"Average weekday consumption: {household_stats['weekday_mean'].mean():.2f} kWh")
print(f"Average weekend consumption: {household_stats['weekend_mean'].mean():.2f} kWh")
print(f"Average weekend/weekday ratio: {household_stats['weekend_ratio'].mean():.3f}")

# Visualize weekend ratio distribution
plt.figure(figsize=(12, 6))
plt.hist(household_stats['weekend_ratio'], bins=50, edgecolor='black', alpha=0.7)
plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Equal (ratio=1.0)')
plt.axvline(household_stats['weekend_ratio'].median(), color='green', linestyle='--', 
            linewidth=2, label=f'Median: {household_stats["weekend_ratio"].median():.3f}')
plt.xlabel('Weekend/Weekday Consumption Ratio')
plt.ylabel('Number of Households')
plt.title('Weekend vs Weekday Consumption Ratio Distribution')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Aggregate consumption by month
print("\n" + "="*80)
print("MONTHLY ANALYSIS")
print("="*80)

monthly_data = aggregate_by_month(train_data)

# Average consumption by month (across all households)
monthly_mean = monthly_data.mean(axis=0)
print("\nAverage Consumption by Month:")
print(monthly_mean)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Line plot
axes[0].plot(monthly_mean.index, monthly_mean.values, marker='o', linewidth=2, markersize=8)
axes[0].fill_between(range(len(monthly_mean)), monthly_mean.values, alpha=0.3)
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Average Consumption (kWh/day)')
axes[0].set_title('Average Daily Consumption by Month')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Box plot
monthly_data.boxplot(ax=axes[1])
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Consumption (kWh/day)')
axes[1].set_title('Consumption Distribution by Month')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Seasonal analysis (Winter vs Summer)
winter_months = ['Jan', 'Feb', 'Dec']
summer_months = ['Jun', 'Jul', 'Aug']

# Filter months that exist in our data
winter_cols = [m for m in winter_months if m in monthly_data.columns]
summer_cols = [m for m in summer_months if m in monthly_data.columns]

household_stats['winter_mean'] = monthly_data[winter_cols].mean(axis=1)
household_stats['summer_mean'] = monthly_data[summer_cols].mean(axis=1)
household_stats['winter_summer_ratio'] = household_stats['winter_mean'] / (household_stats['summer_mean'] + 1e-10)

print("\n" + "="*80)
print("SEASONAL PATTERNS (Winter vs Summer)")
print("="*80)
print(f"Average winter consumption: {household_stats['winter_mean'].mean():.2f} kWh")
print(f"Average summer consumption: {household_stats['summer_mean'].mean():.2f} kWh")
print(f"Average winter/summer ratio: {household_stats['winter_summer_ratio'].mean():.3f}")

# Categorize households by seasonal pattern
seasonal_categories = {
    'Winter-dominant (>1.2)': len(household_stats[household_stats['winter_summer_ratio'] > 1.2]),
    'Balanced (0.8-1.2)': len(household_stats[(household_stats['winter_summer_ratio'] >= 0.8) & 
                                             (household_stats['winter_summer_ratio'] <= 1.2)]),
    'Summer-dominant (<0.8)': len(household_stats[household_stats['winter_summer_ratio'] < 0.8])
}

print("\nSeasonal Pattern Categories:")
for category, count in seasonal_categories.items():
    print(f"  {category}: {count} households ({count/len(household_stats)*100:.1f}%)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram of winter/summer ratio
axes[0].hist(household_stats['winter_summer_ratio'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(1.0, color='red', linestyle='--', linewidth=2, label='Equal (ratio=1.0)')
axes[0].axvline(household_stats['winter_summer_ratio'].median(), color='green', linestyle='--',
                linewidth=2, label=f'Median: {household_stats["winter_summer_ratio"].median():.3f}')
axes[0].set_xlabel('Winter/Summer Consumption Ratio')
axes[0].set_ylabel('Number of Households')
axes[0].set_title('Seasonal Consumption Ratio Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Scatter plot: Winter vs Summer
axes[1].scatter(household_stats['summer_mean'], household_stats['winter_mean'], alpha=0.5, s=10)
axes[1].plot([0, household_stats['summer_mean'].max()], [0, household_stats['summer_mean'].max()], 
             'r--', linewidth=2, label='Equal consumption')
axes[1].set_xlabel('Summer Mean Consumption (kWh/day)')
axes[1].set_ylabel('Winter Mean Consumption (kWh/day)')
axes[1].set_title('Winter vs Summer Consumption')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Trend analysis for sample households
print("\n" + "="*80)
print("TREND ANALYSIS")
print("="*80)

# Analyze trends for sample households
trend_results = {}
for hh_id in sample_households[:5]:  # Analyze first 5 sample households
    trend_results[hh_id] = detect_trend(train_data, hh_id)

print("\nTrend Analysis for Sample Households:")
for hh_id, trend in trend_results.items():
    print(f"\nHousehold {hh_id}:")
    print(f"  Slope: {trend['slope']:.4f} kWh/day")
    print(f"  R²: {trend['r_squared']:.4f}")
    print(f"  Direction: {trend['trend_direction']}")

# Analyze trends for ALL households
all_trends = []
for hh_id in train_data.index[:1000]:  # Sample 1000 households for speed
    trend = detect_trend(train_data, hh_id)
    all_trends.append(trend['slope'])

all_trends = np.array(all_trends)

trend_categories = {
    'Increasing': np.sum(all_trends > 0.01),
    'Stable': np.sum((all_trends >= -0.01) & (all_trends <= 0.01)),
    'Decreasing': np.sum(all_trends < -0.01)
}

print("\nOverall Trend Distribution (1000 households sample):")
for category, count in trend_categories.items():
    print(f"  {category}: {count} households ({count/len(all_trends)*100:.1f}%)")

# Visualize trend distribution
plt.figure(figsize=(12, 6))
plt.hist(all_trends, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--', linewidth=2, label='No trend')
plt.axvline(np.median(all_trends), color='green', linestyle='--', linewidth=2, 
            label=f'Median: {np.median(all_trends):.4f}')
plt.xlabel('Trend Slope (kWh/day per day)')
plt.ylabel('Number of Households')
plt.title('Distribution of Consumption Trends')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## 4. Initial Clustering Insights

### Task 1.4.1 - 1.4.3: Preliminary consumption groups

In [ ]:
# Create correlation matrix for random sample of households
np.random.seed(RANDOM_SEED)
correlation_sample = np.random.choice(train_data.index, size=50, replace=False)
correlation_data = train_data.loc[correlation_sample].T  # Transpose: days × households

corr_matrix = correlation_data.corr()

# Visualize correlation matrix
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, vmin=-1, vmax=1, 
            square=True, linewidths=0.1, cbar_kws={'label': 'Correlation'})
plt.title('Correlation Matrix: 50 Random Households', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("CORRELATION ANALYSIS")
print("="*80)
print(f"Average correlation: {corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)].mean():.3f}")
print(f"Max correlation: {corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)].max():.3f}")
print(f"Min correlation: {corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)].min():.3f}")

In [ ]:
# Categorize households based on consumption patterns
household_categories = categorize_households(household_stats)

print("\n" + "="*80)
print("PRELIMINARY CONSUMPTION GROUPS")
print("="*80)
print("\nCategory Distribution:")
print(household_categories.value_counts())
print("\nPercentages:")
print((household_categories.value_counts() / len(household_categories) * 100).round(2))

# Add categories to stats DataFrame
household_stats['category'] = household_categories

In [ ]:
# Visualize categories
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Category distribution (pie chart)
category_counts = household_categories.value_counts()
axes[0, 0].pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%', startangle=90)
axes[0, 0].set_title('Household Category Distribution')

# 2. Mean consumption by category (box plot)
category_data = [household_stats[household_stats['category'] == cat]['mean'].values 
                 for cat in category_counts.index]
axes[0, 1].boxplot(category_data, labels=category_counts.index)
axes[0, 1].set_ylabel('Mean Consumption (kWh/day)')
axes[0, 1].set_title('Mean Consumption by Category')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Scatter: Mean vs CV colored by category
for cat in category_counts.index:
    cat_data = household_stats[household_stats['category'] == cat]
    axes[1, 0].scatter(cat_data['mean'], cat_data['cv'], label=cat, alpha=0.6, s=20)
axes[1, 0].set_xlabel('Mean Consumption (kWh/day)')
axes[1, 0].set_ylabel('Coefficient of Variation')
axes[1, 0].set_title('Consumption Patterns by Category')
axes[1, 0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1, 0].grid(True, alpha=0.3)

# 4. Bar chart: Average statistics by category
category_stats = household_stats.groupby('category')['mean'].mean().sort_values(ascending=False)
axes[1, 1].barh(category_stats.index, category_stats.values, edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Average Mean Consumption (kWh/day)')
axes[1, 1].set_title('Average Consumption by Category')
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
# Sample time series from each category
fig, axes = plt.subplots(len(category_counts), 1, figsize=(18, len(category_counts)*3))
fig.suptitle('Representative Time Series by Category', fontsize=16, fontweight='bold')

for idx, cat in enumerate(category_counts.index):
    cat_households = household_stats[household_stats['category'] == cat].index
    
    # Sample 3 households from this category
    sample_size = min(3, len(cat_households))
    sample_ids = np.random.choice(cat_households, size=sample_size, replace=False)
    
    for hh_id in sample_ids:
        series = train_data.loc[hh_id]
        axes[idx].plot(series.values, alpha=0.7, linewidth=1, label=f'HH {hh_id}')
    
    axes[idx].set_title(f'{cat} (n={len(cat_households)})', fontweight='bold')
    axes[idx].set_xlabel('Day of Year')
    axes[idx].set_ylabel('Consumption (kWh)')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Summary & Key Insights

In [ ]:
print("\n" + "="*80)
print("EDA SUMMARY & KEY INSIGHTS")
print("="*80)

print("\n📊 DATA OVERVIEW:")
print(f"  • Total households: {len(train_data):,}")
print(f"  • Training period: 365 days (2023)")
print(f"  • Test period: 366 days (2024)")
print(f"  • Total observations: {train_data.size:,}")

print("\n📈 CONSUMPTION STATISTICS:")
print(f"  • Overall mean: {household_stats['mean'].mean():.2f} kWh/day")
print(f"  • Overall median: {household_stats['mean'].median():.2f} kWh/day")
print(f"  • Range: {household_stats['mean'].min():.2f} - {household_stats['mean'].max():.2f} kWh/day")
print(f"  • Standard deviation: {household_stats['mean'].std():.2f} kWh/day")

print("\n⚠️ DATA QUALITY:")
print(f"  • Missing values: {train_validation['missing_values']} (0.00%)")
print(f"  • Zero consumption days: {train_validation['zero_values']:,} ({train_validation['zero_values']/train_data.size*100:.2f}%)")
print(f"  • Households with zeros: {len(households_with_zeros):,} ({len(households_with_zeros)/len(household_stats)*100:.1f}%)")

print("\n🔍 OUTLIERS:")
print(f"  • High consumption outliers: {len(high_outliers_iqr)} households")
print(f"  • Low consumption outliers: {len(low_outliers_iqr)} households")

print("\n📅 TEMPORAL PATTERNS:")
print(f"  • Average weekday consumption: {household_stats['weekday_mean'].mean():.2f} kWh")
print(f"  • Average weekend consumption: {household_stats['weekend_mean'].mean():.2f} kWh")
print(f"  • Weekend/weekday ratio: {household_stats['weekend_ratio'].mean():.3f}")
print(f"  • Average winter consumption: {household_stats['winter_mean'].mean():.2f} kWh")
print(f"  • Average summer consumption: {household_stats['summer_mean'].mean():.2f} kWh")
print(f"  • Winter/summer ratio: {household_stats['winter_summer_ratio'].mean():.3f}")

print("\n🏘️ PRELIMINARY CATEGORIES:")
for cat, count in category_counts.items():
    pct = count / len(household_categories) * 100
    avg_consumption = household_stats[household_stats['category'] == cat]['mean'].mean()
    print(f"  • {cat}: {count:,} households ({pct:.1f}%) - Avg: {avg_consumption:.2f} kWh/day")

print("\n💡 KEY FINDINGS:")
print("  1. Wide variation in consumption patterns (CV ranges from very stable to highly variable)")
print("  2. Significant zero-consumption periods suggest vacant or intermittent occupancy")
print(f"  3. {'Winter' if household_stats['winter_summer_ratio'].mean() > 1 else 'Summer'}-dominant consumption pattern overall")
print("  4. Clear distinction between household categories based on consumption level and stability")
print("  5. Moderate correlation between households suggests clustering potential")

print("\n📝 RECOMMENDATIONS FOR NEXT PHASES:")
print("  • Handle zero-consumption days in preprocessing (imputation or flagging)")
print("  • Consider seasonal features for forecasting models")
print("  • Use day-of-week features (weekday/weekend distinction)")
print("  • Treat outlier households separately or normalize them")
print("  • Explore clustering based on consumption level, variability, and seasonal patterns")
print("  • Consider separate models for stable vs variable consumers")

print("\n" + "="*80)
print("✅ EDA COMPLETE - Ready for Phase 2 (Preprocessing & Feature Engineering)")
print("="*80)

In [ ]:
# Save household statistics for future use
output_path = '../results/household_stats_eda.csv'
household_stats.to_csv(output_path)
print(f"\n💾 Saved household statistics to: {output_path}")
print(f"   Shape: {household_stats.shape}")
print(f"   Columns: {list(household_stats.columns)}")

---
## End of EDA Notebook

**Next Steps:** Proceed to Phase 2 - Data Preprocessing & Feature Engineering  
**Date Completed:** March 12, 2026  
**Status:** ✅ All Phase 1 tasks completed